# Batch Deployment Scoring - Inline 方式

本 notebook 示範如何使用 **Inline 方式**對 watsonx.ai 上的 batch deployment 進行預測。

📖 **參考文件**：[Creating batch deployments](https://www.ibm.com/docs/en/cloud-paks/cp-data/5.1.x?topic=assets-creating-batch-deployments)

---

## 📋 資料輸入方式

watsonx.ai 支援兩種 batch scoring 的資料輸入方式：

1. **Inline 方式**（本 notebook 使用）
2. **Reference 方式**（Data Asset 或 COS）

📖 **參考文件**：[Data sources for scoring batch deployments](https://www.ibm.com/docs/en/cloud-paks/cp-data/5.1.x?topic=deployments-data-sources-scoring-batch)

---

## 🎯 本 Notebook 使用 Inline 方式

本範例使用 **Inline 方式**，可以一次處理多筆資料（2 個樣本）。

### 流程概述
1. 匯入套件
2. 連接到 watsonx.ai
3. 設定 Deployment Space
4. 設定 Deployment ID
5. 驗證 Deployment 狀態
6. 讀取 Scoring 資料
7. 建立 Batch Scoring Job
8. 監控 Job 執行狀態
9. 取得 Scoring 結果
10. 儲存結果到檔案

---

## 🛠️ ibm-watsonx-ai Python Library
https://ibm.github.io/watsonx-ai-python-sdk/

---

## Step 1: 匯入套件

In [1]:
# Import required libraries
import os
import json
import numpy as np
import pandas as pd
from importlib.metadata import version
from ibm_watsonx_ai import Credentials, APIClient
import getpass
from datetime import datetime, timezone, timedelta
import time
import warnings
warnings.filterwarnings('ignore')

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

# Get package versions
np_version = version('numpy')
pd_version = version('pandas')
watsonx_version = version('ibm-watsonx-ai')
cos_version = version('ibm-cos-sdk')

print("="*80)
print("PACKAGE VERSIONS")
print("="*80)
print(f"✅ NumPy version: {np_version}")
print(f"✅ Pandas version: {pd_version}")
print(f"✅ IBM watsonx.ai version: {watsonx_version}")
print(f"✅ IBM COS SDK version: {cos_version}")
print("\n✅ All libraries imported successfully")

PACKAGE VERSIONS
✅ NumPy version: 1.26.4
✅ Pandas version: 2.3.3
✅ IBM watsonx.ai version: 1.5.3
✅ IBM COS SDK version: 2.14.3

✅ All libraries imported successfully


---

## Step 2: 連接到 watsonx.ai

In [2]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

username = 'PASTE YOUR USERNAME HERE'
url = 'https://us-south.ml.cloud.ibm.com'

print("✅ Configuration loaded")

✅ Configuration loaded


### 選擇您的環境：
- **IBM Cloud**: 執行下一個 cell
- **CP4D (on-prem)**: 執行再下一個 cell

#### CP4D

In [ ]:
# CP4D (on-prem)

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.1"
)

client = APIClient(credentials)

print("✅ Connected to Watson Machine Learning")
print(f"   Version: {client.version}")

#### IBM Cloud

In [ ]:
# IBM Cloud

credentials = Credentials(
    url=url,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

client = APIClient(credentials)

print("✅ Connected to Watson Machine Learning")
print(f"   Version: {client.version}")

✅ Connected to Watson Machine Learning
   Version: 1.5.3


---

## Step 3: 設定 Deployment Space

In [4]:
# Project & Space IDs (get from watsonx.ai)

# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

PROJECT_ID = "c4d2e318-f9d4-47c7-9482-f313463b22ea"  # Replace with your project ID
SPACE_ID = "165e735f-11f6-4879-a6fd-99cfecf44f37"  # Replace with your deployment space ID

print("✅ Configuration loaded")
print(f"   PROJECT_ID: {PROJECT_ID}")
print(f"   SPACE_ID: {SPACE_ID}")

✅ Configuration loaded
   PROJECT_ID: c4d2e318-f9d4-47c7-9482-f313463b22ea
   SPACE_ID: 165e735f-11f6-4879-a6fd-99cfecf44f37


### 綁定 Deployment Space

**重要**：Batch deployments 必須部署在 **Deployment Space** 中，請執行下方的 "Link default space" cell。

（Project 與 Space 不能同時綁定）

#### Link default project

In [13]:
# client.set.default_project(PROJECT_ID)

# print(f"✅ Default project set to: {PROJECT_ID}")
# print(f"\nProject details:")
# project_details = client.projects.get_details(PROJECT_ID)
# print(f"   Name: {project_details['entity']['name']}")

#### Link default space

In [5]:
client.set.default_space(SPACE_ID)

print(f"✅ Default space set to: {SPACE_ID}")
print(f"\nSpace details:")
space_details = client.spaces.get_details(SPACE_ID)
print(f"   Name: {space_details['entity']['name']}")

✅ Default space set to: 165e735f-11f6-4879-a6fd-99cfecf44f37

Space details:
   Name: taishin_ai_gov_enablement_and_poc_dev_ds


---

## Step 4: 設定 Deployment ID

In [6]:
# ============================================================================
# CONFIGURATION - Update with your deployment ID
# ============================================================================

DEPLOYMENT_ID = "4db3549b-5529-4be3-86c2-509ad42a3123"  # Replace with your deployment ID

print(f"✅ Deployment ID configured: {DEPLOYMENT_ID}")

✅ Deployment ID configured: 4db3549b-5529-4be3-86c2-509ad42a3123


---

## Step 5: 驗證 Deployment 狀態

In [8]:
# 取得 deployment 詳細資訊
try:
    deployment_details = client.deployments.get_details(DEPLOYMENT_ID)
    
    print("="*80)
    print("DEPLOYMENT DETAILS")
    print("="*80)
    print(f"✅ Deployment ID: {DEPLOYMENT_ID}")
    print(f"✅ Name: {deployment_details['metadata']['name']}")
    print(f"✅ Type: {deployment_details['entity']['deployed_asset_type']}")
    print(f"✅ Status: {deployment_details['entity']['status']['state']}")
    
    # 檢查狀態
    status = deployment_details['entity']['status']['state']
    if status == 'ready':
        print("\n✅ Deployment is ready for scoring!")
    else:
        print(f"\n⚠️  Warning: Deployment status is '{status}', not 'ready'")
        print("   Please wait for the deployment to be ready before scoring.")
        
except Exception as e:
    print(f"❌ Error getting deployment details: {str(e)}")
    print("   Please check:")
    print("   1. Deployment ID is correct")
    print("   2. You have access to the deployment space")
    print("   3. The deployment exists in the specified space")

DEPLOYMENT DETAILS
✅ Deployment ID: 4db3549b-5529-4be3-86c2-509ad42a3123
✅ Name: SavedModel_0326-1246_batch
✅ Type: model
✅ Status: ready

✅ Deployment is ready for scoring!


---

## Step 6: 讀取 Scoring 資料

In [19]:
# 設定 scoring 資料來源檔案

# ============================================================================
# CONFIGURATION - Update with your scoring data file
# ============================================================================

scoring_data_file = '2_batch_savedmodel_scoring_input_data_example.json'

print(f"✅ Scoring data file configured: {scoring_data_file}")

✅ Scoring data file configured: 2_batch_savedmodel_scoring_input_data_example.json


In [ ]:
# 讀取 scoring 資料
with open(scoring_data_file, 'r') as f:
    scoring_payload = json.load(f)

print("="*80)
print("SCORING DATA LOADED")
print("="*80)
print(f"✅ File: {scoring_data_file}")
print(f"✅ Number of input features: {len(scoring_payload['input_data'])}")

# 顯示第一個樣本的數量
first_input = scoring_payload['input_data'][0]
num_samples = len(first_input['values'])
print(f"✅ Number of samples: {num_samples}")

# 顯示所有輸入特徵的名稱
print(f"\n📊 Input features:")
for i, input_data in enumerate(scoring_payload['input_data'], 1):
    feature_id = input_data['id']
    values_shape = np.array(input_data['values']).shape
    print(f"   {i:2d}. {feature_id:35s} - shape: {values_shape}")

SCORING DATA LOADED
✅ File: 2_batch_savedmodel_scoring_input_data_example.json
✅ Number of input features: 32
✅ Number of samples: 2

📊 Input features:
    1. input_x_o_MD_BRN_CONTACTS           - shape: (2, 16)
    2. input_x_c_1_MD_BRN_CONTACTS         - shape: (2, 1)
    3. input_x_c_2_MD_BRN_CONTACTS         - shape: (2, 1)
    4. input_x_c_3_MD_BRN_CONTACTS         - shape: (2, 1)
    5. input_x_c_4_MD_BRN_CONTACTS         - shape: (2, 1)
    6. input_x_c_5_MD_BRN_CONTACTS         - shape: (2, 1)
    7. input_x_c_6_MD_BRN_CONTACTS         - shape: (2, 1)
    8. input_x_c_7_MD_BRN_CONTACTS         - shape: (2, 1)
    9. input_x_c_8_MD_BRN_CONTACTS         - shape: (2, 1)
   10. input_x_t_MD_CASH_FLOW              - shape: (2, 12, 22)
   11. input_x_t_MD_CUS_EXCH               - shape: (2, 12, 12)
   12. input_x_t_MD_CUS_LOAN               - shape: (2, 12, 24)
   13. input_x_o_MD_CUS_RISK               - shape: (2, 1)
   14. input_x_c_1_MD_CUS_RISK             - shape: (2, 1)
   15.

---

## Step 7: 建立 Batch Scoring Job

In [ ]:
# 準備 job payload (input data)
job_payload = {
    client.deployments.ScoringMetaNames.INPUT_DATA: scoring_payload['input_data']
}

print("="*80)
print("CREATING BATCH SCORING JOB")
print("="*80)

try:
    # 建立 batch job
    job_details = client.deployments.create_job(DEPLOYMENT_ID, job_payload)
    job_id = client.deployments.get_job_uid(job_details)
    
    print(f"✅ Batch job created successfully!")
    print(f"✅ Job ID: {job_id}")
    print(f"\n⏳ Job is now queued for execution...")
    
except Exception as e:
    print(f"❌ Error creating batch job: {str(e)}")
    job_id = None

CREATING BATCH SCORING JOB
✅ Batch job created successfully!
✅ Job ID: 7d8e6bd2-59ef-4fb0-a673-b3cb212ae2da

⏳ Job is now queued for execution...


---

## Step 8: 監控 Job 執行狀態

In [11]:
if job_id:
    print("="*80)
    print("MONITORING JOB STATUS")
    print("="*80)
    print(f"Job ID: {job_id}")
    print("\n⏳ Waiting for job to complete...\n")
    
    # 監控 job 執行狀態
    check_count = 0
    state = None
    job_status = None
    while True:
        try:
            job_status = client.deployments.get_job_details(job_id)
            state = job_status['entity']['scoring']['status']['state']
            
            # 顯示當前狀態
            check_count += 1
            timestamp = datetime.now(TZ_UTC8).strftime('%Y-%m-%d %H:%M:%S')
            print(f"[{timestamp}] Check #{check_count}: Job status = {state}")
            
            # 檢查是否完成
            if state in ['completed', 'failed', 'canceled', 'error']:
                break
            
            # 等待 10 秒後再次檢查
            time.sleep(3)
            
        except Exception as e:
            print(f"❌ Error checking job status: {str(e)}")
            break
    
    # 顯示最終狀態
    print("\n" + "="*80)
    if state == 'completed':
        print("✅ BATCH SCORING COMPLETED SUCCESSFULLY!")
    elif state == 'failed':
        print("❌ BATCH SCORING FAILED!")
        if job_status and 'message' in job_status['entity']['scoring']['status']:
            print(f"   Error message: {job_status['entity']['scoring']['status']['message']}")
    elif state == 'canceled':
        print("⚠️  BATCH SCORING WAS CANCELED")
    elif state:
        print(f"⚠️  BATCH SCORING ENDED WITH STATUS: {state}")
    else:
        print("⚠️  BATCH SCORING STATUS UNKNOWN (monitoring interrupted)")
    print("="*80)
else:
    print("❌ No job ID available. Please create a job first.")
    state = None

MONITORING JOB STATUS
Job ID: 7d8e6bd2-59ef-4fb0-a673-b3cb212ae2da

⏳ Waiting for job to complete...

[2026-04-01 09:54:40] Check #1: Job status = queued
[2026-04-01 09:54:43] Check #2: Job status = queued
[2026-04-01 09:54:48] Check #3: Job status = running
[2026-04-01 09:54:52] Check #4: Job status = completed

✅ BATCH SCORING COMPLETED SUCCESSFULLY!


---

## Step 9: 取得 Scoring 結果

In [12]:
if job_id and state == 'completed':
    print("="*80)
    print("SCORING RESULTS")
    print("="*80)
    
    try:
        # 取得完整的 job 詳細資訊
        results = client.deployments.get_job_details(job_id)
        
        # 提取預測結果
        if 'scoring' in results['entity'] and 'predictions' in results['entity']['scoring']:
            predictions = results['entity']['scoring']['predictions']
            
            print(f"✅ Number of predictions: {len(predictions)}")
            print("\n📊 Prediction Results:\n")
            
            # 顯示每個預測結果
            for i, pred in enumerate(predictions, 1):
                print(f"{'='*60}")
                print(f"Sample {i}:")
                print(f"{'='*60}")
                print(json.dumps(pred, indent=2))
                print()
        else:
            print("⚠️  No predictions found in job results")
            print("\nFull job details:")
            print(json.dumps(results, indent=2))
            
    except Exception as e:
        print(f"❌ Error retrieving results: {str(e)}")
else:
    if not job_id:
        print("❌ No job ID available")
    elif state != 'completed':
        print(f"⚠️  Job did not complete successfully (status: {state})")

SCORING RESULTS
✅ Number of predictions: 3

📊 Prediction Results:

Sample 1:
{
  "id": "y_layers",
  "values": [
    [
      0.4536029100418091,
      0.39778682589530945,
      0.5047555565834045,
      0.434455931186676,
      0.4982425570487976,
      0.41581159830093384,
      0.36640435457229614,
      0.4855055809020996,
      0.5084773302078247,
      0.46940261125564575,
      0.5626829266548157,
      0.4128006398677826,
      0.4748046398162842,
      0.5358611941337585,
      0.34636059403419495,
      0.3874305188655853,
      0.4731336832046509,
      0.6304177045822144,
      0.36508792638778687
    ],
    [
      0.44555822014808655,
      0.4459860026836395,
      0.4023300111293793,
      0.5345574617385864,
      0.41009631752967834,
      0.4173925220966339,
      0.4491363763809204,
      0.4244122803211212,
      0.4479408264160156,
      0.4972156882286072,
      0.44331446290016174,
      0.38625332713127136,
      0.41578900814056396,
      0.5525185465812683,
 

---

## Step 10: 儲存結果到檔案

In [34]:
if job_id and state == 'completed':
    # 產生輸出檔案名稱（包含時間戳記）
    timestamp = datetime.now(TZ_UTC8).strftime('%Y%m%d_%H%M%S')
    output_file = f"batch_scoring_results_{timestamp}.json"
    
    try:
        # 儲存完整結果
        with open(output_file, 'w') as f:
            json.dump(results, f, indent=2)
        
        print("="*80)
        print("RESULTS SAVED")
        print("="*80)
        print(f"✅ Results saved to: {output_file}")
        print(f"✅ File size: {os.path.getsize(output_file):,} bytes")
        
        # 也儲存僅包含預測結果的檔案
        predictions_file = f"batch_predictions_{timestamp}.json"
        with open(predictions_file, 'w') as f:
            json.dump(predictions, f, indent=2)
        
        print(f"✅ Predictions only saved to: {predictions_file}")
        print(f"✅ File size: {os.path.getsize(predictions_file):,} bytes")
        
    except Exception as e:
        print(f"❌ Error saving results: {str(e)}")
else:
    print("⚠️  No results to save (job did not complete successfully)")

RESULTS SAVED
✅ Results saved to: batch_scoring_results_20260401_013812.json
✅ File size: 123,840 bytes
✅ Predictions only saved to: batch_predictions_20260401_013812.json
✅ File size: 2,220 bytes


---

## ✅ 完成！

### 執行摘要

您已完成以下步驟：
1. ✅ 連接到 watsonx.ai
2. ✅ 設定 Deployment Space
3. ✅ 讀取 scoring 資料
4. ✅ 驗證 deployment 狀態
5. ✅ 建立 batch scoring job
6. ✅ 監控 job 執行
7. ✅ 取得並儲存結果

### 📁 相關檔案
- **輸入資料**: `2_batch_savedmodel_scoring_input_data_example.json`
- **完整結果**: `batch_scoring_results_<timestamp>.json`
- **預測結果**: `batch_predictions_<timestamp>.json`

---

## ❓ 常見問題

### Q: Job 一直處於 'queued' 狀態？
- 檢查 deployment 的資源配置
- 確認沒有其他 job 正在執行
- 等待更長時間（大型模型可能需要較長時間）

### Q: 出現 "Deployment not found" 錯誤？
- 確認 Deployment ID 正確
- 確認已綁定正確的 Space ID
- 確認您有該 Space 的存取權限

### Q: 輸入資料格式錯誤？
- 確認 JSON 格式與模型輸入格式完全匹配
- 檢查所有必要的 input_data 欄位都存在
- 驗證資料型別（數值、陣列維度等）